In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


# 1. 데이터 로딩
cust_df = pd.read_csv("../data/santander-customer-satisfaction/train.csv", encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score

RANDOM_STATE = 42


def preprocess(df):
    """최종 채택된 1~5단계 전처리만 적용 (369 -> 145개 피처)."""
    y = df['TARGET']
    X = df.drop(columns=['ID', 'TARGET'])

    df['var3'] = df['var3'].replace(-999999, 2)

    dup_cols = X.columns[X.T.duplicated()].tolist()
    X = X.drop(columns=dup_cols)

    stds = X.std()
    X = X.drop(columns=stds[stds == 0].index.tolist())

    num_rows = X.shape[0]
    sparse_cols = [c for c in X.columns if (X[c] == 0).sum() / num_rows >= 0.99]
    X = X.drop(columns=sparse_cols)

    X['var38'] = np.log1p(X['var38'])
    X['var15_below_23'] = (X['var15'] < 23).astype(int)
    X['var15_bin'] = pd.cut(X['var15'], bins=5, labels=False).astype(int)

    return X, y

In [3]:
raw_df = pd.read_csv('../data/santander-customer-satisfaction/train.csv', encoding='latin-1')

X, y = preprocess(raw_df)
print(f"최종 피처 수: {X.shape[1]}")

최종 피처 수: 145


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2, random_state=42)

In [6]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [7]:
# RandomForest로 학습
# RandomForest object 생성
rf_clf = RandomForestClassifier(random_state=42)
# 학습
rf_clf.fit(X_train, y_train)

# 예측값
pred = rf_clf.predict(X_test)

# 원래 답, 예측값 비교
accuracy= accuracy_score(y_test, pred)

# 예측 확률
pred_proba = rf_clf.predict_proba(X_test)[:,-1]

# ROC-AUC
roc_auc = roc_auc_score(y_test, pred_proba)


print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy)) # 출력
print('랜덤 포레스트 ROC-AUC: {0:.4f}'.format(roc_auc))

랜덤 포레스트 정확도: 0.9527
랜덤 포레스트 ROC-AUC: 0.7556


In [8]:
from sklearn.model_selection import cross_val_score
rf_default = RandomForestClassifier(
    n_estimators=500, random_state=42, n_jobs=-1
)
s = cross_val_score(rf_default, X_train, y_train, cv=2, scoring='roc_auc')
print(f'{s.mean():.4f} ± {s.std():.4f}')

0.7629 ± 0.0052


In [9]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators':[500],
    'max_depth' : [28, 32, 40],  
    'min_samples_split' : [22, 26, 30]
}
# RandomForestClassifier 객체 생성 후 GridSearchCV 수행
rf_clf = RandomForestClassifier(random_state=42, n_jobs=-1)  # n_jobs: 병렬처리할 cpu 개수 지정
grid_cv = GridSearchCV(rf_clf , param_grid=params, scoring='roc_auc', cv=2, n_jobs=-1 )
grid_cv.fit(X_train , y_train)


print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
print('최고 AUC: {0:.4f}'.format(grid_cv.best_score_))

# 1차 최적 하이퍼 파라미터:
#  {'max_depth': 16, 'min_samples_leaf': 8, 'min_samples_split': 8, 'n_estimators': 100}
# 최고 AUC: 0.8119

# 2차 최적 하이퍼 파라미터:
#  {'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 16, 'n_estimators': 100}
# 최고 AUC: 0.8130

# 3차 최적 하이퍼 파라미터:
#  {'max_depth': 28, 'min_samples_leaf': 1, 'min_samples_split': 22, 'n_estimators': 100}
# 최고 AUC: 0.8171

# 4차 최적 하이퍼 파라미터:
#  {'max_depth': 28, 'min_samples_split': 30, 'n_estimators': 500}
# 최고 AUC: 0.8217

최적 하이퍼 파라미터:
 {'max_depth': 28, 'min_samples_split': 30, 'n_estimators': 500}
최고 AUC: 0.8217


In [10]:
rf_clf = RandomForestClassifier(random_state=42, max_depth=28, min_samples_split=30, n_estimators=500)
rf_clf.fit(X_train, y_train)
pred = rf_clf.predict(X_test)
accuracy= accuracy_score(y_test, pred)

pred_proba = rf_clf.predict_proba(X_test)[:,-1]

# ROC-AUC
roc_auc = roc_auc_score(y_test, pred_proba)

print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy)) # 출력
print('랜덤 포레스트 ROC-AUC: {0:.4f}'.format(roc_auc))

랜덤 포레스트 정확도: 0.9601
랜덤 포레스트 ROC-AUC: 0.8329
